# NN playground — ONE Blackjack Lab

Load a trained checkpoint, evaluate it, look inside (strategy charts, learned index plays, bet head vs exact
analytics) and kick off training. Background and design decisions: `docs/NN.md`.

Setup once: `uv sync --group notebook`, then pick the project's `.venv` as the kernel.
Evaluations use all CPU cores (multiprocessing); the round counts below are kept small so every cell runs in
seconds to a minute — raise them for publication-quality numbers.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

from onebj.config import load_config
from onebj.ai.agent import NNAgent, list_checkpoints, load_training_log
from onebj.ai import analysis, evaluate

cfg = load_config()
for c in list_checkpoints():
    print(c["name"], c["created"], {k: v for k, v in (c["metrics"] or {}).items() if isinstance(v, float)})

In [ ]:
CKPT = "nn_v1"
agent = NNAgent.load(CKPT, cfg, trace=True)
desc = agent.describe()
print([(l["name"], l["size"]) for l in desc["layers"]], [(h["name"], h["labels"]) for h in desc["heads"]])
print(json.dumps(desc["training"]["metrics"], indent=1))

## Training curves
The log has one row every `log_every` steps (training loss) and one row per generation with the evaluation metrics.

In [ ]:
log = load_training_log(CKPT)
def series(key):
    rows = [r for r in log if isinstance(r.get(key), (int, float))]
    return [r["step"] for r in rows], [r[key] for r in rows]

fig, ax = plt.subplots(1, 4, figsize=(17, 3.4))
for k in ("loss_play", "loss_bet"):
    ax[0].plot(*series(k), label=k)
ax[0].set_yscale("log"); ax[0].legend(); ax[0].set_title("training loss")
for k, lab in (("eval_ev", "NN"), ("eval_ev_basic", "basic (same cards)")):
    ax[1].plot(*series(k), "o-", label=lab)
ax[1].legend(); ax[1].set_title("flat-bet EV per round (paired eval)")
x, y = series("gain_vs_basic"); _, se = series("gain_se")
ax[2].errorbar(x, y, yerr=2 * np.array(se), fmt="o-"); ax[2].axhline(0, color="grey", lw=0.5)
ax[2].set_title("gain vs basic per round (±2 SE)")
for k in ("regret", "regret_basic"):
    ax[3].plot(*series(k), "o-", label=k)
ax[3].legend(); ax[3].set_title("solver regret per decision")
for a in ax: a.set_xlabel("step")
plt.tight_layout()

## Evaluate
`evaluate_agent` plays the full game (bets, side bets, insurance, play); `duplicate_eval` replays every round with
basic strategy on the same cards, so the play-quality difference has a ~10× smaller standard error.
Agent specs: `"nn_v1"` (Kelly + side bets + insurance), `"nn_v1:flat"` (play only), `"basic"`, `"hilo"`,
`"hilo_nosides"`, `"solver"` (composition-dependent DP oracle, slow).

In [ ]:
results = [evaluate.evaluate_agent(s, cfg, rounds=200_000, seed=1) for s in (CKPT, CKPT + ":flat", "basic", "hilo")]
print(evaluate.markdown_table(results))
print(evaluate.cashout_table(results))

In [ ]:
dup = evaluate.duplicate_eval(CKPT, "basic", cfg, rounds=300_000, seed=2)
print(f"NN play {dup['ev_a']:+.5f} vs basic {dup['ev_b']:+.5f} per round → gain {dup['diff']:+.5f} ± {dup['diff_se']:.5f}")

## Strategy charts
Network actions by player total × dealer up card for a synthetic shoe (3 decks unseen) at Hi-Lo true count −3, 0, +3. `C` = cash out. Cells that differ from basic strategy are outlined.

In [ ]:
ACTION_COLORS = {"stand": "#e9c46a", "hit": "#90be6d", "double": "#4d908e", "split": "#577590", "cashout": "#f94144"}
UPS = list("23456789TA")
basic = analysis.basic_chart(cfg)

def plot_chart(ax, chart, title):
    rows = [("H" + r, c, basic["hard"][r]) for r, c in chart["hard"].items()] + \
           [("S" + r, c, basic["soft"][r]) for r, c in chart["soft"].items()] + \
           [("P" + r, c, basic["pairs"][r]) for r, c in chart["pairs"].items()]
    for i, (lab, cells, ref) in enumerate(rows):
        for j, a in enumerate(cells):
            ax.add_patch(plt.Rectangle((j, i), 1, 1, color=ACTION_COLORS[a]))
            ax.text(j + 0.5, i + 0.55, a[0].upper(), ha="center", va="center", fontsize=7)
            if a != ref[j]:
                ax.add_patch(plt.Rectangle((j + 0.04, i + 0.04), 0.92, 0.92, fill=False, lw=1.5, ec="black"))
    ax.set_xlim(0, 10); ax.set_ylim(len(rows), 0)
    ax.set_xticks(np.arange(10) + 0.5, UPS); ax.set_yticks(np.arange(len(rows)) + 0.5, [r[0] for r in rows], fontsize=7)
    ax.set_title(title)

fig, ax = plt.subplots(1, 3, figsize=(15, 11))
for a, tc in zip(ax, (-3, 0, 3)):
    plot_chart(a, analysis.strategy_chart(agent, tc=tc, decks=3.0), f"TC {tc:+d}")
plt.tight_layout()

## Learned index plays
The true count where the network switches between two actions, next to the Hi-Lo (Illustrious 18) index.

In [ ]:
for r in analysis.deviation_table(agent):
    nn = f"{r['nn_index']:+.1f}" if r["nn_index"] is not None else r["status"]
    print(f"{r['name']:<16} Hi-Lo {r['hilo_index']:+.0f}   NN {nn}")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.4))
for a, (total, up, act, vs, title) in zip(ax[:2], ((16, 9, "stand", "hit", "16 v T: Q(stand) − Q(hit)"),
                                                   (15, 9, "cashout", "hit", "15 v T: Q(cashout) − Q(hit)"))):
    sc = analysis.index_scan(agent, total, False, up, act, vs)
    a.plot(sc["tcs"], sc["diffs"]); a.axhline(0, color="grey", lw=0.5); a.set_title(title); a.set_xlabel("Hi-Lo TC")
ins = analysis.insurance_scan(agent)
ax[2].plot(ins["tcs"], ins["predicted"], label="NN"); ax[2].plot(ins["tcs"], ins["exact"], "--", label="exact")
ax[2].axhline(0, color="grey", lw=0.5); ax[2].legend(); ax[2].set_title("insurance EV"); ax[2].set_xlabel("Hi-Lo TC")
plt.tight_layout()

## Bet head vs exact analytics
Predicted EV per unit stake of each bet before the deal vs the exact composition EV (`onebj.analytics`; for `main`
the EoR-based basic-strategy *estimate*, which ignores the network's better play and cash-outs).

In [ ]:
bv = analysis.bet_validation(agent, n=6000, seed=3)
labels = ["main", "perfect_pairs", "twenty_one_plus_three", "crazy_7", "bust_bonus", "insurance"]
fig, ax = plt.subplots(1, 6, figsize=(20, 3.4))
for i, (a, lab) in enumerate(zip(ax, labels)):
    e, p = bv["exact"][:, i], bv["pred"][:, i]
    a.scatter(e, p, s=2, alpha=0.3)
    lo, hi = min(e.min(), p.min()), max(e.max(), p.max())
    a.plot([lo, hi], [lo, hi], "k--", lw=0.7); a.axhline(0, lw=0.4, color="grey"); a.axvline(0, lw=0.4, color="grey")
    s = bv["stats"][lab]
    a.set_title(f"{lab}\nr={s['corr']:.3f}  MAE={s['mae']:.4f}", fontsize=9); a.set_xlabel("exact")
ax[0].set_ylabel("predicted")
plt.tight_layout()

## One decision, traced
This is the JSON the NN panel renders (SPEC §5).

In [ ]:
from onebj.engine import Table, Phase
t = Table(cfg, seed=7)
bets, bet_trace = agent.decide_bets(t.observation())
t.place_bets(bets)
while t.phase is Phase.INSURANCE:
    t.act("no_insurance")
if t.phase is Phase.PLAYER:
    action, tr = agent.decide_action(t.observation())
    print({k: tr[k] for k in ("chosen", "outputs", "reference", "extra")})
    print([(f["name"], f["raw"], f["value"]) for f in tr["features"] if f["value"] != 0][:20])

## Train
The default run (`uv run python -m onebj.ai.train --name nn_v1`) takes ~45 min on an M4 with 10 cores. A quick run
for experiments (~1 min) is below; every `TrainConfig` field can be overridden, e.g. the feature spec, the trunk
size, data per generation, the play-target mode (`"solver"`, `"blend"`, `"mc"`) and the generator knobs.
Multiprocessing works from the notebook because the workers live in importable modules.

In [ ]:
from onebj.ai.train import TrainConfig, train

tc = TrainConfig(
    generations=[{"play": 200_000, "bet": 80_000}, {"play": 100_000, "bet": 40_000}],
    targets_per_generation=["solver", "blend"],
    steps_per_generation=[3000, 2000],
    eval_rounds=200_000, bench_states=10_000,
)
# meta = train("nb_experiment", tc, cfg, out_dir="checkpoints")   # uncomment to run (~1-2 min)